# Food Production Challenge 2 - Baseline Submission

This notebook provides a simple baseline for **Food Production Challenge 2: Quality Control Prediction**.

**Goal**: Predict `qc_pass` (0/1) for each production lot
**Metric**: Macro-F1 Score - Higher is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular lot data with a simple Random Forest classifier.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="your-api-key-here",        # Get from your team dashboard
    team_name="your-team-name-here"     # Your exact team name
)

# Load data from PVC paths
print("📂 Loading Food Production Challenge 2 data...")

# Load lot data
train_lots = pd.read_csv("/home/jovyan/shared/datasets/FoodProduction/lots_train.csv")
test_lots = pd.read_csv("/home/jovyan/shared/datasets/FoodProduction/lots_test.csv")

print(f"✅ Data loaded:")
print(f"   Train lots: {train_lots.shape}")
print(f"   Test lots: {test_lots.shape}")
print(f"   Train columns: {list(train_lots.columns)}")
print(f"   Test columns: {list(test_lots.columns)}")


In [ ]:
# 2. Tabular-Only Baseline Model and Predictions

# From data inspection - lots columns:
# lot_id, sku_id, site_id, cook_temp_F, cook_time_min, chill_time_min, acidification_pH, line_speed, sanitation_gap_min, inspector_note, qc_pass (train only)

# Select numeric production features for baseline
lot_features = ['sku_id', 'site_id', 'cook_temp_F', 'cook_time_min', 'chill_time_min', 'acidification_pH', 'line_speed', 'sanitation_gap_min']
print(f"📊 Using lot features: {lot_features}")

# Prepare training data
X_train = train_lots[lot_features].fillna(0)
y_train = train_lots['qc_pass']  # Binary target (0/1)

# Prepare test data
X_test = test_lots[lot_features].fillna(0)

# Train simple Random Forest baseline
print("🤖 Training Random Forest classifier...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
predictions = model.predict(X_test)

# Create submission file (format: lot_id,qc_pass)
submission_df = pd.DataFrame({
    'lot_id': test_lots['lot_id'],
    'qc_pass': predictions
})

# Save predictions
submission_df.to_csv("foodproduction_challenge2_predictions.csv", index=False)
print(f"✅ Predictions saved: {submission_df.shape[0]} predictions")
print(f"   Preview: {submission_df.head(3)}")
print(f"   QC pass rate: {predictions.mean():.3f} ({predictions.sum()} passed out of {len(predictions)})")


In [ ]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("FoodProduction", 2, "foodproduction_challenge2_predictions.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. Move on to Food Production Challenge 3!")
